Checking PSD of R for DINO v2 on COCO dataset

PSD of R <=> eigenvalues of R >= 0

In [1]:
import os
import torch
import torch.nn.functional as F
from torchvision import transforms
import cv2

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [3]:
# load model
model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').to(device)

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:01<00:00, 63.6MB/s]


In [4]:
model.eval()

DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(14, 14), stride=(14, 14))
    (norm): Identity()
  )
  (blocks): ModuleList(
    (0-11): 12 x NestedTensorBlock(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): MemEffAttention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (drop_path1): Identity()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=1536, out_features=384, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
      (ls2): LayerScale()
      (drop_path2): Identity()
    )
  )
  (norm): LayerNorm((384,), eps=1e-06, elementwise_affi

In [5]:
!wget http://images.cocodataset.org/zips/test2017.zip
!unzip -q test2017.zip

--2026-03-28 14:29:15--  http://images.cocodataset.org/zips/test2017.zip
Resolving images.cocodataset.org (images.cocodataset.org)... 3.5.30.44, 52.216.53.17, 54.231.226.49, ...
Connecting to images.cocodataset.org (images.cocodataset.org)|3.5.30.44|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6646970404 (6.2G) [application/zip]
Saving to: ‘test2017.zip’

test2017.zip        100%[===================>]   6.19G  37.6MB/s    in 3m 18s  

2026-03-28 14:32:33 (32.1 MB/s) - ‘test2017.zip’ saved [6646970404/6646970404]



In [6]:
test_dir = "test2017"
files = os.listdir(test_dir)
len(files)

40670

In [7]:
def preprocessing(image):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image_size = (224, 224)
    blob = cv2.dnn.blobFromImage(image, 1.0/255, image_size, swapRB=True, crop=False)
    blob[0][0] = (blob[0][0] - 0.485)/0.229
    blob[0][1] = (blob[0][1] - 0.456)/0.224
    blob[0][2] = (blob[0][2] - 0.406)/0.225
    return torch.tensor(blob)

In [8]:
result = model.get_intermediate_layers(preprocessing(cv2.imread(os.path.join(test_dir, files[0]))).to(device))

In [9]:
result[0].shape

torch.Size([1, 256, 384])

In [10]:
import math
math.sqrt(256)

16.0

In [12]:
m = 1e9
for i, f in enumerate(files):
    img = cv2.imread(os.path.join(test_dir, f))
    blob = preprocessing(img).to(device)
    result = model.get_intermediate_layers(blob)
    feats = result[0][0][1:]
    W = torch.matmul(feats, feats.T)
    minW = W.min().item()
    d = torch.matmul(feats,feats.sum(dim=0).unsqueeze(1))
    mind = d.min().item()
    if mind < m:
        m = mind
        print(i,'d',mind,'W',minW,"!!!!!" if mind <= 0 else "")
    D = torch.diag(d.squeeze(-1))
    eigenvalues, eigenvectors = torch.lobpcg(A=D-W, k=2, B=D, largest=False)
    if eigenvalues[0] < -1e-5:
        print(i,'eig',eigenvalues[0].item(),"!!!!!" if eigenvalues[0].item() <= 0 else "")
    if i % 500 == 0:
        print(i,minW,mind,eigenvalues[0].item(),eigenvalues[1].item())

0 d 80425.90625 W -280.7264099121094 
0 -280.7264099121094 80425.90625 1.3347565186450083e-07 0.37041550874710083
1 d 76247.859375 W -247.97491455078125 
2 d 66010.5 W -230.88821411132812 
7 d 52493.0625 W -265.59698486328125 
9 d 48925.28515625 W -197.27952575683594 
47 d 38005.6015625 W -387.6012878417969 
132 d 36593.3671875 W -273.5559997558594 
236 d 23008.984375 W -452.3614196777344 
290 d 21290.42578125 W -317.7375793457031 
500 -134.81732177734375 121917.703125 -2.7864240692565545e-08 0.5724563002586365
1000 -185.07357788085938 81331.984375 -3.5195458991665873e-08 0.47669869661331177
1500 0.1696910858154297 117180.328125 2.443526057049894e-07 0.6956911683082581
1571 d 19212.25 W -320.3192443847656 
2000 -190.00491333007812 89676.59375 1.960921736099408e-07 0.5728280544281006
2500 -179.54159545898438 76279.609375 2.0814430001792061e-07 0.4384141266345978
2903 d 12002.8203125 W -396.8282470703125 
3000 -71.34467315673828 122732.4296875 -1.8739640950116154e-07 0.6546827554702759
3

Though the values of W are also negative, d is always well-defined for all samples. However, it seems that R might not be PSD (-0.044 is barely zero plus a numeric error).